# Градиентный бустинг. Классификатор. Домашка вариант 2

In [188]:
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_wine, load_iris, load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, r2_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingClassifier

import xgboost as xgb
import lightgbm as lgb

sns.set_theme(style="whitegrid")

In [189]:
class GBMClassifier:
  def __init__(self, n_estimators= 400, learning_rate = 0.05, max_depth=7, l2 = 1.0, seed=42):
    self.n_estimators = n_estimators
    self.learning_rate = learning_rate
    self.max_depth = max_depth
    self.l2 = l2
    self.seed = seed
    self.trees = []
    self.n_classes = None
    self.base_predictions = None

    # вернет вероятности
  def softmax(self, scores):
    exps = np.exp(scores - np.max(scores, axis=1, keepdims=True))
    return exps / np.sum(exps, axis=1, keepdims=True)



  def fit(self, X, y):
    self.n_classes = len(np.unique(y))
    n_samples = X.shape[0]

    self.base_predictions = np.zeros(self.n_classes)
    for k in range(self.n_classes):
      # с защитой от нуля в логарифме
      self.base_predictions[k] = np.log(np.mean(y == k) + 1e-9)

    scores = np.zeros((n_samples, self.n_classes))
    scores += self.base_predictions

    # в каждом из списокв внутри списка будут лежать деревья заточенные под данный класс
    self.trees = [[] for _ in range(self.n_classes)]

    for _ in range(self.n_estimators):
      probabilities = self.softmax(scores)

      for k in range(self.n_classes):
        target = (y == k).astype(float)
        antigrad = target - probabilities[:, k]
        # для регуляризации. гемини сказал ленивый вариант
        hessians = probabilities[:, k] * (1 - probabilities[:, k])
        tree = DecisionTreeRegressor(max_depth=self.max_depth)
        tree.fit(X, antigrad)
        # гессиан берем средний по дереву
        reg_coeff = 1.0/(np.mean(hessians) + self.l2)
        tree.tree_.value[:] *= reg_coeff
        prediction = tree.predict(X)
        scores[:, k] += self.learning_rate * prediction
        self.trees[k].append(tree)

  def predict_proba(self, X):
    n_samples = X.shape[0]
    scores = np.zeros((n_samples, self.n_classes))
    scores += self.base_predictions

    for k in range(self.n_classes):
      for tree in self.trees[k]:
        scores[:, k] += tree.predict(X) * self.learning_rate

    return self.softmax(scores)

  def predict(self, X):
    probabilities = self.predict_proba(X)
    return np.argmax(probabilities, axis=1)


# ПРОВЕРКА ТОГО ЧТО ПОЛУЧИЛОСЬ
решил проверить на трех разных датасетах из sklearn. При том для каждого конечного результа по метрикам обучал по 50 моделей каждого вида(150 на датасет всего) каждый раз при рандомных разбиениях датасета. выводил среднее по R2 для каждой модели и стандартное отклонение. Иначе при фиксированном сиде я мог бы выбрать самые классные разбиения когда моя модель превосходили модели XGBoost или Sklearn. есть скрины. GridSearch для подбора параметров не делал. параметры всем моделям давал одинаковые в рамках одного датасета.

In [190]:
data = load_wine()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

my_r2_scores = []
sk_r2_scores = []
xgb_r2_scores = []

n_iterations = 50

for i in range(n_iterations):
    # я не прописывал сид. хотел 50 рандомных разбиений
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
    my_model = GBMClassifier(n_estimators=100, learning_rate=0.3, max_depth=4)
    sk_model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.3, max_depth=4)
    xgb_model = xgb.XGBClassifier(n_estimators=100, learning_rate=0.3, max_depth=4)

    my_model.fit(X_train, y_train)
    sk_model.fit(X_train, y_train)
    xgb_model.fit(X_train, y_train)

    my_r2_scores.append(r2_score(y_test, my_model.predict(X_test)))
    sk_r2_scores.append(r2_score(y_test, sk_model.predict(X_test)))
    xgb_r2_scores.append(r2_score(y_test, xgb_model.predict(X_test)))

print("\n Results on WINE DATASET:")
print(f"my_model  R2(mean): {np.mean(my_r2_scores):.4f} (Std: {np.std(my_r2_scores):.4f})")
print(f"sk_model  R2(mean): {np.mean(sk_r2_scores):.4f} (Std: {np.std(sk_r2_scores):.4f})")
print(f"xgb_model R2(mean): {np.mean(xgb_r2_scores):.4f} (Std: {np.std(xgb_r2_scores):.4f})")


 Results on WINE DATASET:
my_model  R2(mean): 0.8868 (Std: 0.0646)
sk_model  R2(mean): 0.8964 (Std: 0.0679)
xgb_model R2(mean): 0.9188 (Std: 0.0734)


In [196]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

my_r2_scores = []
sk_r2_scores = []
xgb_r2_scores = []

n_iterations = 50

for i in range(n_iterations):
    # я не прописывал сид. хотел 50 рандомных разбиений
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
    my_model = GBMClassifier(n_estimators=100, learning_rate=0.1, max_depth=4)
    sk_model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=4)
    xgb_model = xgb.XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=4)

    my_model.fit(X_train, y_train)
    sk_model.fit(X_train, y_train)
    xgb_model.fit(X_train, y_train)

    my_r2_scores.append(r2_score(y_test, my_model.predict(X_test)))
    sk_r2_scores.append(r2_score(y_test, sk_model.predict(X_test)))
    xgb_r2_scores.append(r2_score(y_test, xgb_model.predict(X_test)))

print("\n Results on BREAST CANCER DATASET:")
print(f"my_model  R2(mean): {np.mean(my_r2_scores):.4f} (Std: {np.std(my_r2_scores):.4f})")
print(f"sk_model  R2(mean): {np.mean(sk_r2_scores):.4f} (Std: {np.std(sk_r2_scores):.4f})")
print(f"xgb_model R2(mean): {np.mean(xgb_r2_scores):.4f} (Std: {np.std(xgb_r2_scores):.4f})")


 Results on BREAST CANCER DATASET:
my_model  R2(mean): 0.7887 (Std: 0.0743)
sk_model  R2(mean): 0.8333 (Std: 0.0738)
xgb_model R2(mean): 0.8582 (Std: 0.0611)


In [192]:
data = load_iris()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

my_r2_scores = []
sk_r2_scores = []
xgb_r2_scores = []

n_iterations = 50

for i in range(n_iterations):
    # я не прописывал сид. хотел 50 рандомных разбиений
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
    my_model = GBMClassifier(n_estimators=100, learning_rate=0.3, max_depth=4)
    sk_model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.3, max_depth=4)
    xgb_model = xgb.XGBClassifier(n_estimators=100, learning_rate=0.3, max_depth=4)

    my_model.fit(X_train, y_train)
    sk_model.fit(X_train, y_train)
    xgb_model.fit(X_train, y_train)

    my_r2_scores.append(r2_score(y_test, my_model.predict(X_test)))
    sk_r2_scores.append(r2_score(y_test, sk_model.predict(X_test)))
    xgb_r2_scores.append(r2_score(y_test, xgb_model.predict(X_test)))

print("\n Results on IRIS DATASET:")
print(f"my_model  R2(mean): {np.mean(my_r2_scores):.4f} (Std: {np.std(my_r2_scores):.4f})")
print(f"sk_model  R2(mean): {np.mean(sk_r2_scores):.4f} (Std: {np.std(sk_r2_scores):.4f})")
print(f"xgb_model R2(mean): {np.mean(xgb_r2_scores):.4f} (Std: {np.std(xgb_r2_scores):.4f})")


 Results on IRIS DATASET:
my_model  R2(mean): 0.9176 (Std: 0.0634)
sk_model  R2(mean): 0.9285 (Std: 0.0719)
xgb_model R2(mean): 0.9042 (Std: 0.0669)


# РЕЗУЛЬТАТЫ
можно видеть выше. в целом критерию соответствует. отклонение на ирисах и ввине на кастомной модели самое малое. только на раке груди есть отставание более заметное.